# 2608.27372 — Ellipsoid-Fitting Phase Boundary

**Engineering Statements companion notebook — v4**

This version separates solver output from validated computational readings.

For every solver-reported feasible point, the notebook checks the returned matrix \(R\) directly through

\[
\max_i |x_i^\top R x_i - 1|
\]

and

\[
\lambda_{\min}(R).
\]

The workflow is

\[
\boxed{
\text{solver status}
\rightarrow
\text{constraint validation}
\rightarrow
\text{computational reading}
}
\]

with readings:

\[
\boxed{
\mathrm{SAT}
\neq
\mathrm{UNSAT}
\neq
\mathrm{UNRESOLVED}
}
\]

Source: arXiv:2608.27372  
Engineering Statement: `statements/2608-27372.yaml`

## 1. Runtime mode and validation tolerances

The default is the report-scale Gaussian experiment:

\[
d=40,\qquad 8\text{ trials per density}.
\]

A solver-reported feasible matrix is accepted as a validated SAT reading only if both checks pass:

\[
\max_i |x_i^\top R x_i - 1|
\le \varepsilon_{\mathrm{eq}}
\]

and

\[
\lambda_{\min}(R)
\ge -\varepsilon_{\mathrm{psd}}.
\]

The tolerances are explicit so the computational reading remains inspectable.

In [ ]:
MODE = "paper"   # "fast" or "paper"

if MODE == "fast":
    D = 12
    ALPHAS = [0.10, 0.15, 0.20, 0.23, 0.24, 0.25, 0.26, 0.27, 0.30, 0.35]
    TRIALS = 3
else:
    D = 40
    ALPHAS = [0.10, 0.15, 0.20, 0.225, 0.24, 0.25, 0.26, 0.275, 0.30, 0.35, 0.40]
    TRIALS = 8

SEED = 260827372

EQ_TOL = 1e-4
PSD_TOL = 1e-6

print({
    "mode": MODE,
    "d": D,
    "trials_per_density": TRIALS,
    "alpha_values": ALPHAS,
    "eq_tolerance": EQ_TOL,
    "psd_tolerance": PSD_TOL,
})

## 2. Install and import dependencies

In [ ]:
!pip -q install pyyaml cvxpy

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import yaml

print("cvxpy:", cp.__version__)
print("installed solvers:", cp.installed_solvers())

## 3. Load the Engineering Statement

The repository YAML is primary. The embedded fallback keeps the notebook portable.

In [ ]:
STATEMENT_PATH = Path("../statements/2608-27372.yaml")

fallback_yaml = """
id: 2608-27372
title: Universality and Sharp Thresholds for Ellipsoid Fitting
source:
  paper: https://arxiv.org/abs/2608.27372
objective: >
  Specify the ellipsoid-fitting problem through its constraint-density scaling,
  sharp SAT-UNSAT phase boundary, distribution-dependent threshold,
  and computational readings.
constraints:
  - Random vectors x_1,...,x_n lie in R^d.
  - Seek a positive-semidefinite matrix R such that x_i^T R x_i = 1 for every i.
  - Constraint density is alpha = n / d^2.
  - A sharp SAT-UNSAT phase boundary occurs at alpha_star(kappa).
  - The coordinate distribution enters the phase boundary through kappa = E[x_ij^4].
  - For Gaussian coordinates, kappa = 3 and alpha_star(3) = 1/4.
"""

if STATEMENT_PATH.exists():
    statement = yaml.safe_load(STATEMENT_PATH.read_text())
    print("Loaded:", STATEMENT_PATH)
else:
    statement = yaml.safe_load(fallback_yaml)
    print("Using embedded fallback statement.")

print(statement["title"])
print(statement["objective"].strip())

## 4. Gaussian specification

For Gaussian coordinates,

\[
\kappa=3,
\qquad
\alpha_\star(3)=\frac14.
\]

At \(d=40\),

\[
n\approx \frac{d^2}{4}=400.
\]

In [ ]:
KAPPA_GAUSSIAN = 3.0
ALPHA_STAR = 1.0 / 4.0

print({
    "kappa": KAPPA_GAUSSIAN,
    "alpha_star": ALPHA_STAR,
    "d40_predicted_n": int(ALPHA_STAR * 40**2),
})

## 5. Generate Gaussian instances

In [ ]:
def gaussian_instance(d, n, rng):
    return rng.normal(size=(n, d))

## 6. Solver policy

Each instance is attempted with available solvers in sequence:

```text
CLARABEL
↓ retry if needed
SCS
```

The solver status is preserved, but a feasible status alone does not define the final reading.

A feasible candidate is passed to the validation step.

In [ ]:
FEASIBLE_STATUSES = {
    cp.OPTIMAL,
    cp.OPTIMAL_INACCURATE,
}

INFEASIBLE_STATUSES = {
    cp.INFEASIBLE,
    cp.INFEASIBLE_INACCURATE,
}

def available_solver_order():
    installed = set(cp.installed_solvers())
    preferred = [s for s in ("CLARABEL", "SCS") if s in installed]
    if not preferred:
        raise RuntimeError("Neither CLARABEL nor SCS is available.")
    return preferred

SOLVER_ORDER = available_solver_order()
print("Solver order:", SOLVER_ORDER)

## 7. Direct validation of a candidate matrix

For each solver-returned \(R\), compute:

\[
r_{\max}
=
\max_i |x_i^\top R x_i - 1|
\]

and

\[
\lambda_{\min}(R).
\]

The matrix is accepted as a validated SAT reading when

\[
r_{\max}\le \varepsilon_{\mathrm{eq}}
\]

and

\[
\lambda_{\min}(R)\ge -\varepsilon_{\mathrm{psd}}.
\]

In [ ]:
def validate_candidate(X, R_value, eq_tol=EQ_TOL, psd_tol=PSD_TOL):
    if R_value is None:
        return {
            "valid": False,
            "max_eq_residual": np.nan,
            "min_eigenvalue": np.nan,
            "eq_ok": False,
            "psd_ok": False,
        }

    X = np.asarray(X, dtype=float)
    R = np.asarray(R_value, dtype=float)

    # Symmetrize tiny numerical asymmetry before eigendecomposition.
    R_sym = 0.5 * (R + R.T)

    quadratic_values = np.einsum("ni,ij,nj->n", X, R_sym, X)
    max_eq_residual = float(np.max(np.abs(quadratic_values - 1.0)))
    min_eigenvalue = float(np.linalg.eigvalsh(R_sym)[0])

    eq_ok = max_eq_residual <= eq_tol
    psd_ok = min_eigenvalue >= -psd_tol

    return {
        "valid": bool(eq_ok and psd_ok),
        "max_eq_residual": max_eq_residual,
        "min_eigenvalue": min_eigenvalue,
        "eq_ok": bool(eq_ok),
        "psd_ok": bool(psd_ok),
    }

## 8. Ellipsoid-fitting SDP with retry and validation

The final reading policy is:

```text
solver says feasible
    ↓
validate R
    ↓
valid constraints → SAT
failed validation → retry next solver
no validated feasible result → UNRESOLVED

solver says infeasible
    ↓
UNSAT
```

This keeps solver status and validated reading distinct.

In [ ]:
def solve_with_solver(problem, solver):
    if solver == "SCS":
        problem.solve(
            solver=solver,
            verbose=False,
            eps=1e-5,
            max_iters=50000,
        )
    else:
        problem.solve(
            solver=solver,
            verbose=False,
        )

def ellipsoid_fit_reading(X, solver_order=SOLVER_ORDER):
    X = np.asarray(X, dtype=float)
    n, d = X.shape

    R = cp.Variable((d, d), symmetric=True)
    constraints = [R >> 0]
    constraints += [cp.quad_form(X[i], R) == 1 for i in range(n)]
    problem = cp.Problem(cp.Minimize(0), constraints)

    attempts = []

    for solver in solver_order:
        try:
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter("always")
                solve_with_solver(problem, solver)

            warning_text = " | ".join(str(w.message) for w in caught) if caught else None
            status = problem.status

            if status in FEASIBLE_STATUSES:
                validation = validate_candidate(X, R.value)

                attempts.append({
                    "solver": solver,
                    "status": str(status),
                    "solver_reading": "FEASIBLE",
                    "validated_reading": "SAT" if validation["valid"] else "UNRESOLVED",
                    "max_eq_residual": validation["max_eq_residual"],
                    "min_eigenvalue": validation["min_eigenvalue"],
                    "eq_ok": validation["eq_ok"],
                    "psd_ok": validation["psd_ok"],
                    "warning": warning_text,
                    "exception": None,
                })

                if validation["valid"]:
                    return {
                        "reading": "SAT",
                        "final_solver": solver,
                        "final_status": str(status),
                        "max_eq_residual": validation["max_eq_residual"],
                        "min_eigenvalue": validation["min_eigenvalue"],
                        "attempts": attempts,
                    }

                # Feasible solver status but failed direct validation:
                # try the next solver if one is available.
                continue

            if status in INFEASIBLE_STATUSES:
                attempts.append({
                    "solver": solver,
                    "status": str(status),
                    "solver_reading": "INFEASIBLE",
                    "validated_reading": "UNSAT",
                    "max_eq_residual": np.nan,
                    "min_eigenvalue": np.nan,
                    "eq_ok": None,
                    "psd_ok": None,
                    "warning": warning_text,
                    "exception": None,
                })

                return {
                    "reading": "UNSAT",
                    "final_solver": solver,
                    "final_status": str(status),
                    "max_eq_residual": np.nan,
                    "min_eigenvalue": np.nan,
                    "attempts": attempts,
                }

            attempts.append({
                "solver": solver,
                "status": str(status),
                "solver_reading": "UNKNOWN",
                "validated_reading": "UNRESOLVED",
                "max_eq_residual": np.nan,
                "min_eigenvalue": np.nan,
                "eq_ok": None,
                "psd_ok": None,
                "warning": warning_text,
                "exception": None,
            })

        except Exception as exc:
            attempts.append({
                "solver": solver,
                "status": None,
                "solver_reading": "ERROR",
                "validated_reading": "UNRESOLVED",
                "max_eq_residual": np.nan,
                "min_eigenvalue": np.nan,
                "eq_ok": None,
                "psd_ok": None,
                "warning": None,
                "exception": f"{type(exc).__name__}: {exc}",
            })

    final = attempts[-1] if attempts else {}

    return {
        "reading": "UNRESOLVED",
        "final_solver": final.get("solver"),
        "final_status": final.get("status"),
        "max_eq_residual": final.get("max_eq_residual", np.nan),
        "min_eigenvalue": final.get("min_eigenvalue", np.nan),
        "attempts": attempts,
    }

## 9. Quick validation reading

This verifies that the notebook produces a solver result and a direct constraint check.

In [ ]:
rng = np.random.default_rng(SEED)
X_check = gaussian_instance(d=6, n=6, rng=rng)
check = ellipsoid_fit_reading(X_check)

print("reading:", check["reading"])
print("final solver:", check["final_solver"])
print("final status:", check["final_status"])
print("max equality residual:", check["max_eq_residual"])
print("minimum eigenvalue:", check["min_eigenvalue"])
print("attempts:")
for attempt in check["attempts"]:
    print(" ", attempt)

## 10. Sweep constraint density

At each value of

\[
\alpha=\frac{n}{d^2},
\]

the notebook records:

- validated SAT,
- solver-supported UNSAT,
- unresolved,
- equality residuals,
- minimum eigenvalues,
- retries,
- warnings,
- complete solver attempt history.

In [ ]:
def sweep_gaussian(d, alphas, trials, seed):
    rng = np.random.default_rng(seed)
    summary_rows = []
    trial_rows = []

    total = len(alphas) * trials
    completed = 0

    for alpha_target in alphas:
        n = max(1, int(round(alpha_target * d**2)))
        alpha = n / d**2

        counts = {
            "SAT": 0,
            "UNSAT": 0,
            "UNRESOLVED": 0,
        }

        sat_residuals = []
        sat_min_eigs = []

        for trial in range(1, trials + 1):
            X = gaussian_instance(d=d, n=n, rng=rng)
            result = ellipsoid_fit_reading(X)

            reading = result["reading"]
            counts[reading] += 1
            completed += 1

            retry_used = len(result["attempts"]) > 1

            if reading == "SAT":
                sat_residuals.append(result["max_eq_residual"])
                sat_min_eigs.append(result["min_eigenvalue"])

            residual_text = (
                f"{result['max_eq_residual']:.2e}"
                if np.isfinite(result["max_eq_residual"])
                else "—"
            )
            eig_text = (
                f"{result['min_eigenvalue']:.2e}"
                if np.isfinite(result["min_eigenvalue"])
                else "—"
            )

            print(
                f"{completed:>3}/{total}  "
                f"d={d:>2} n={n:>4} alpha={alpha:.4f}  "
                f"trial={trial}/{trials}  "
                f"{reading:<10}  "
                f"solver={result['final_solver']}  "
                f"status={result['final_status']}  "
                f"residual={residual_text}  "
                f"lambda_min={eig_text}  "
                f"retry={retry_used}"
            )

            trial_rows.append({
                "d": d,
                "n": n,
                "alpha": alpha,
                "trial": trial,
                "reading": reading,
                "final_solver": result["final_solver"],
                "final_status": result["final_status"],
                "max_eq_residual": result["max_eq_residual"],
                "min_eigenvalue": result["min_eigenvalue"],
                "retry_used": retry_used,
                "attempts_json": json.dumps(result["attempts"]),
            })

        resolved = counts["SAT"] + counts["UNSAT"]

        summary_rows.append({
            "d": d,
            "n": n,
            "alpha": alpha,
            "trials": trials,
            "sat_count": counts["SAT"],
            "unsat_count": counts["UNSAT"],
            "unresolved_count": counts["UNRESOLVED"],
            "resolved_count": resolved,
            "fraction_resolved": resolved / trials,
            "fraction_unresolved": counts["UNRESOLVED"] / trials,
            "fraction_sat_among_resolved": (
                counts["SAT"] / resolved if resolved > 0 else np.nan
            ),
            "max_sat_eq_residual": (
                max(sat_residuals) if sat_residuals else np.nan
            ),
            "min_sat_eigenvalue": (
                min(sat_min_eigs) if sat_min_eigs else np.nan
            ),
        })

    return pd.DataFrame(summary_rows), pd.DataFrame(trial_rows)

readings, trial_log = sweep_gaussian(
    d=D,
    alphas=ALPHAS,
    trials=TRIALS,
    seed=SEED,
)

## 11. Summary table

In [ ]:
display(
    readings[
        [
            "d",
            "n",
            "alpha",
            "trials",
            "sat_count",
            "unsat_count",
            "unresolved_count",
            "fraction_resolved",
            "fraction_sat_among_resolved",
            "max_sat_eq_residual",
            "min_sat_eigenvalue",
        ]
    ]
)

## 12. Trial-level validation log

This exposes solver status and direct validation measurements for every trial.

In [ ]:
display(
    trial_log[
        [
            "d",
            "n",
            "alpha",
            "trial",
            "reading",
            "final_solver",
            "final_status",
            "max_eq_residual",
            "min_eigenvalue",
            "retry_used",
        ]
    ]
)

## 13. Plot validated SAT readings

This plot uses only final validated readings.

Unresolved trials are excluded from the SAT fraction and shown separately in the following plot.

In [ ]:
fig_sat, ax = plt.subplots(figsize=(9, 5.5))

plot_data = readings.dropna(subset=["fraction_sat_among_resolved"])

ax.plot(
    plot_data["alpha"],
    plot_data["fraction_sat_among_resolved"],
    marker="o",
    linewidth=2,
    label="Validated fraction SAT among resolved trials",
)

ax.axvline(
    ALPHA_STAR,
    linestyle="--",
    linewidth=2,
    label=r"Theoretical boundary $\alpha_\star(3)=1/4$",
)

ax.set_xlabel(r"Constraint density $\alpha=n/d^2$")
ax.set_ylabel("Fraction SAT among resolved trials")
ax.set_ylim(-0.05, 1.05)
ax.set_title(
    f"Validated ellipsoid-fitting readings — d={D}, "
    f"{TRIALS} trials per density"
)
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 14. Plot unresolved readings

In [ ]:
fig_unresolved, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(
    readings["alpha"],
    readings["fraction_unresolved"],
    marker="o",
    linewidth=2,
    label="Fraction unresolved",
)

ax.axvline(
    ALPHA_STAR,
    linestyle="--",
    linewidth=2,
    label=r"Theoretical boundary $\alpha_\star(3)=1/4$",
)

ax.set_xlabel(r"Constraint density $\alpha=n/d^2$")
ax.set_ylabel("Fraction unresolved")
ax.set_ylim(-0.05, 1.05)
ax.set_title(
    f"Unresolved computational readings — d={D}"
)
ax.legend()
ax.grid(alpha=0.2)

plt.show()

## 15. Validate the boundary region

This table isolates the readings nearest the Gaussian theoretical boundary.

In [ ]:
boundary_window = readings[
    (readings["alpha"] >= 0.225) &
    (readings["alpha"] <= 0.275)
]

display(
    boundary_window[
        [
            "alpha",
            "n",
            "sat_count",
            "unsat_count",
            "unresolved_count",
            "fraction_sat_among_resolved",
            "max_sat_eq_residual",
            "min_sat_eigenvalue",
        ]
    ]
)

## 16. Compact computational reading

In [ ]:
below = readings[readings["alpha"] < ALPHA_STAR]
at = readings[np.isclose(readings["alpha"], ALPHA_STAR)]
above = readings[readings["alpha"] > ALPHA_STAR]

print("Below theoretical Gaussian boundary:")
print(f"  SAT: {int(below['sat_count'].sum())}")
print(f"  UNSAT: {int(below['unsat_count'].sum())}")
print(f"  unresolved: {int(below['unresolved_count'].sum())}")

print("\nAt theoretical Gaussian boundary:")
print(f"  SAT: {int(at['sat_count'].sum())}")
print(f"  UNSAT: {int(at['unsat_count'].sum())}")
print(f"  unresolved: {int(at['unresolved_count'].sum())}")

print("\nAbove theoretical Gaussian boundary:")
print(f"  SAT: {int(above['sat_count'].sum())}")
print(f"  UNSAT: {int(above['unsat_count'].sum())}")
print(f"  unresolved: {int(above['unresolved_count'].sum())}")

## 17. Save outputs

The summary, full validation log, and both plots are written as repository-ready artifacts.

In [ ]:
OUTPUT_DIR = Path("../outputs/2608-27372")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_summary_v4.csv"
trial_csv = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_trial_log_v4.csv"
sat_png = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_validated_sat_v4.png"
unresolved_png = OUTPUT_DIR / f"gaussian_d{D}_{MODE}_unresolved_v4.png"

readings.to_csv(summary_csv, index=False)
trial_log.to_csv(trial_csv, index=False)

fig_sat.savefig(sat_png, dpi=180, bbox_inches="tight")
fig_unresolved.savefig(unresolved_png, dpi=180, bbox_inches="tight")

print("Saved:")
print(summary_csv)
print(trial_csv)
print(sat_png)
print(unresolved_png)

## 18. Engineering reading

The notebook now distinguishes three layers:

\[
\boxed{
\text{asymptotic specification}
\rightarrow
\text{solver result}
\rightarrow
\text{validated computational reading}
}
\]

For a feasible candidate, the reading is accepted only after checking the actual constraints:

\[
\max_i |x_i^\top R x_i - 1|
\]

and

\[
\lambda_{\min}(R).
\]

For Gaussian coordinates:

\[
\boxed{
\kappa=3
\rightarrow
\alpha_\star(3)=\frac14
\rightarrow
\text{SDP}
\rightarrow
\text{constraint validation}
\rightarrow
\text{SAT / UNSAT / unresolved}
}
\]